In [1]:
# ============================================
# ЯЧЕЙКА 1
# Импорты, конфигурация и загрузка CSV
# ============================================

import os, copy
import math
import time
import random
import threading
from datetime import datetime
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# --------------------------------------------
# КОНФИГУРАЦИЯ
# --------------------------------------------

CSV_FILE = "/content/btc_history.csv"

INPUT_LEN = 100
HORIZON = 10
STEP = 10
FEAT_DIM = 11

TOTAL_CANDLES = 5_000_000

TARGET_PARAMS = 50_000_000

ACC_THRESHOLD = 32.0

W_DIR = 0.6
W_PROX = 0.4

PRETRAIN_EPOCHS = 10
BATCH = 64
LR = 1e-3

FIX_STEPS = 6
FIX_LR = 1e-4
FIX_COOLDOWN = 2

BAD_BUF = 200
MIX_EVERY = 3
MIX_STEPS = 4
MIX_LR = 3e-4

MODEL_FILE = "/content/forecaster_v2.pth"

NUM_WORKERS = 0
LAMBDA_DIR = 0.5  # добавь в конфиг

def compute_loss(pred, y):
    mse = ((pred - y) ** 2 * loss_weights).mean()

    dir_target = torch.sign(y[:, :, 3])
    dir_loss = torch.relu(-pred[:, :, 3] * dir_target).mean()

    return mse + LAMBDA_DIR * dir_loss


# --------------------------------------------
# УСТРОЙСТВО
# --------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("Устройство:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "CUDA память:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print("=" * 60)


# --------------------------------------------
# РАЗМЕР МОДЕЛИ
# --------------------------------------------

def forecaster_dims(target):
    M = 512 if target > 1_000_000 else 256

    a = 9.0
    b = 45.0 + M
    c = 41.0 * M + 40.0 - target

    D = b * b - 4.0 * a * c

    H = int(
        (-b + math.sqrt(D)) / (2.0 * a)
    )

    return max(8, H), M


HIDDEN, H_MID = forecaster_dims(TARGET_PARAMS)


# --------------------------------------------
# ЧТЕНИЕ CSV
# --------------------------------------------

def read_csv_rows(path):

    rows = []

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            parts = line.split(",")

            if len(parts) < 6:
                continue

            try:

                timestamp = int(float(parts[0]))
                o = float(parts[1])
                h = float(parts[2])
                l = float(parts[3])
                c = float(parts[4])
                v = float(parts[5])

                rows.append(
                    (timestamp, o, h, l, c, v)
                )

            except Exception:
                continue

    rows.sort(key=lambda x: x[0])

    return rows


if not os.path.exists(CSV_FILE):
    raise FileNotFoundError(
        f"CSV не найден:\n{CSV_FILE}\n\n"
        "Загрузи btc_history.csv в Colab."
    )


candles = read_csv_rows(CSV_FILE)

print("Свечей загружено:", len(candles))

if len(candles) < INPUT_LEN + HORIZON + 100:
    raise RuntimeError(
        "Недостаточно свечей для обучения."
    )


# --------------------------------------------
# МАССИВЫ
# --------------------------------------------

timestamps = np.asarray(
    [x[0] for x in candles],
    dtype=np.int64
)

o = np.asarray(
    [x[1] for x in candles],
    dtype=np.float32
)

h = np.asarray(
    [x[2] for x in candles],
    dtype=np.float32
)

l = np.asarray(
    [x[3] for x in candles],
    dtype=np.float32
)

c = np.asarray(
    [x[4] for x in candles],
    dtype=np.float32
)

v = np.asarray(
    [x[5] for x in candles],
    dtype=np.float32
)


# --------------------------------------------
# FEATURE ENGINEERING
# --------------------------------------------

def rolling_std(x, window):

    x64 = x.astype(np.float64)

    s1 = np.concatenate(
        ([0.0], np.cumsum(x64))
    )

    s2 = np.concatenate(
        ([0.0], np.cumsum(x64 * x64))
    )

    out = np.zeros(len(x), dtype=np.float64)

    for i in range(len(x)):

        start = max(0, i - window + 1)

        n = i - start + 1

        sum_x = s1[i + 1] - s1[start]
        sum_x2 = s2[i + 1] - s2[start]

        mean = sum_x / n

        var = max(
            0.0,
            sum_x2 / n - mean * mean
        )

        out[i] = math.sqrt(var)

    return out.astype(np.float32)


def build_features_full(o, h, l, c, v):

    n = len(c)

    close_safe = np.maximum(c, 1e-12)

    ret = np.zeros(n, dtype=np.float32)

    ret[1:] = np.log(
        close_safe[1:] /
        close_safe[:-1]
    )

    mom5 = np.zeros(n, dtype=np.float32)
    mom20 = np.zeros(n, dtype=np.float32)

    mom5[5:] = (
        close_safe[5:] /
        close_safe[:-5] - 1.0
    )

    mom20[20:] = (
        close_safe[20:] /
        close_safe[:-20] - 1.0
    )

    mean20 = np.zeros(n, dtype=np.float32)

    cs = np.concatenate(
        ([0.0], np.cumsum(v.astype(np.float64)))
    )

    for i in range(n):

        start = max(0, i - 19)

        mean20[i] = (
            (cs[i + 1] - cs[start]) /
            (i - start + 1)
        )

    volrel = (
        v /
        np.maximum(mean20, 1e-12)
    )

    rng = (
        (h - l) /
        np.maximum(close_safe, 1e-12)
    )

    std20 = rolling_std(ret, 20)
    std100 = rolling_std(ret, 100)

    G = np.zeros(
        (n, 10),
        dtype=np.float32
    )

    G[:, 0] = o / close_safe - 1.0
    G[:, 1] = h / close_safe - 1.0
    G[:, 2] = l / close_safe - 1.0

    G[:, 3] = ret * 1000.0
    G[:, 4] = std20 * 1000.0
    G[:, 5] = std100 * 1000.0

    G[:, 6] = mom5 * 1000.0
    G[:, 7] = mom20 * 1000.0

    G[:, 8] = volrel
    G[:, 9] = rng * 1000.0

    return G


print("Создание признаков...")

G10 = build_features_full(
    o, h, l, c, v
)

# 11-й признак — относительная цена закрытия
G = np.zeros(
    (len(c), FEAT_DIM),
    dtype=np.float32
)

G[:, :10] = G10

G[:, 10] = 0.0

print("Размер признаков:", G.shape)


# --------------------------------------------
# DATASET
# --------------------------------------------

class CandleDataset(Dataset):

    def __init__(
        self,
        G,
        o,
        h,
        l,
        c,
        start,
        end,
        step=15
    ):

        self.G = G
        self.o = o
        self.h = h
        self.l = l
        self.c = c

        self.start = start
        self.end = end
        self.step = step

        max_start = (
            end -
            INPUT_LEN -
            HORIZON
        )

        if max_start < start:
            self.positions = np.empty(
                0,
                dtype=np.int64
            )
        else:
            self.positions = np.arange(
                start,
                max_start + 1,
                step,
                dtype=np.int64
            )

    def __len__(self):
        return len(self.positions)

    def __getitem__(self, idx):

        p = int(self.positions[idx])

        base = max(
            float(self.c[p + INPUT_LEN - 1]),
            1e-12
        )

        x = self.G[
            p:p + INPUT_LEN
        ].copy()

        # 11-й признак:
        x[:, 10] = (
            self.c[
                p:p + INPUT_LEN
            ] / base - 1.0
        )

        target_start = (
            p + INPUT_LEN
        )

        target_end = (
            target_start + HORIZON
        )

        y = np.stack(
            [
                self.o[target_start:target_end],
                self.h[target_start:target_end],
                self.l[target_start:target_end],
                self.c[target_start:target_end],
            ],
            axis=1
        ).astype(np.float32)

        y /= base
        y -= 1.0

        return (
            torch.from_numpy(x),
            torch.from_numpy(y)
        )


# --------------------------------------------
# ВРЕМЕННОЕ РАЗДЕЛЕНИЕ
# --------------------------------------------

N = len(c)

split = int(N * 0.8)

# небольшой purge-gap между train и test
gap = INPUT_LEN + HORIZON

train_end = split - gap

test_start = split

train_dataset = CandleDataset(
    G,
    o,
    h,
    l,
    c,
    0,
    train_end,
    STEP
)

test_dataset = CandleDataset(
    G,
    o,
    h,
    l,
    c,
    test_start,
    N,
    STEP
)

print()
print("TRAIN samples:", len(train_dataset))
print("TEST samples :", len(test_dataset))
print("TRAIN candles:", train_end)
print("TEST candles :", N - test_start)

print()

Устройство: cpu


FileNotFoundError: CSV не найден:
/content/btc_history.csv

Загрузи btc_history.csv в Colab.

In [ ]:
# ============================================
# ЯЧЕЙКА 2
# Модель, метрики, предобучение и онлайн-обучение
# ============================================

class Forecaster(nn.Module):

    def __init__(
        self,
        in_dim=FEAT_DIM,
        hidden=HIDDEN,
        mid=H_MID,
        horizon=HORIZON
    ):

        super().__init__()

        self.gru = nn.GRU(
            input_size=in_dim,
            hidden_size=hidden,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )

        self.drop = nn.Dropout(0.2)

        self.head = nn.Sequential(
            nn.Linear(hidden, mid),
            nn.ReLU(),
            nn.Linear(mid, horizon * 4)
        )

        self.horizon = horizon

    def forward(self, x):
        out, _ = self.gru(x)
        last = self.drop(out[:, -1])
        y = self.head(last)
        return y.view(-1, self.horizon, 4)



model = Forecaster().to(device)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Параметров модели: "
    f"{parameter_count:,}"
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

loss_weights = torch.tensor(
    [1.0, 1.0, 1.0, 2.0],
    device=device
).view(1, 1, 4)


# --------------------------------------------
# МЕТРИКА
# --------------------------------------------

def accuracy_metric(pred, actual):

    pc = pred[:, :, 3]
    ac = actual[:, :, 3]

    # направление КАЖДОГО из 10 шагов относительно точки входа
    pred_dir = (
        torch.sign(pc) == torch.sign(ac)
    ).float().mean(dim=1)

    pred_range = (
        pred[:, :, 1].max(dim=1).values -
        pred[:, :, 2].min(dim=1).values
    )

    actual_range = (
        actual[:, :, 1].max(dim=1).values -
        actual[:, :, 2].min(dim=1).values
    )

    prox = torch.clamp(
        1.0 - torch.abs(pred_range - actual_range) /
        torch.clamp(actual_range, min=1e-6),
        0.0, 1.0
    )

    score = 100.0 * (W_DIR * pred_dir + W_PROX * prox)

    return score


# --------------------------------------------
# BRAIN
# --------------------------------------------

class Brain:

    def __init__(self):

        self.model = model
        self.optimizer = optimizer

        self.bad_buffer = []

        self.fix_count = 0

        self.last_accuracy = None

        self.phase = "IDLE"

        self.stop_event = threading.Event()

        self.lock = threading.Lock()

    def stop(self):

        self.stop_event.set()

        print("⛔ Запрос остановки отправлен.")

    def reset_stop(self):

        self.stop_event.clear()

    def predict(self, x):

        self.model.eval()

        if isinstance(x, np.ndarray):

            x = torch.from_numpy(
                x
            ).float()

        x = x.unsqueeze(0).to(device)

        with torch.no_grad():

            pred = self.model(x)

        return pred[0].cpu().numpy()

    def train_batch(
        self,
        xb,
        yb,
        learning_rate
    ):

        for group in self.optimizer.param_groups:
            group["lr"] = learning_rate

        self.model.train()

        xb = xb.to(
            device,
            non_blocking=True
        )

        yb = yb.to(
            device,
            non_blocking=True
        )

        self.optimizer.zero_grad(
            set_to_none=True
        )

        pred = self.model(xb)

        loss = compute_loss(pred, y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            self.model.parameters(),
            5.0
        )

        self.optimizer.step()

        return float(loss.item())

    def evaluate(
        self,
        dataset,
        max_samples=300
    ):

        if len(dataset) == 0:
            return 0.0

        loader = DataLoader(
            dataset,
            batch_size=BATCH,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available()
        )

        self.model.eval()

        scores = []
        count = 0

        with torch.no_grad():

            for xb, yb in loader:

                if count >= max_samples:
                    break

                remaining = (
                    max_samples - count
                )

                xb = xb[:remaining]
                yb = yb[:remaining]

                xb = xb.to(device)
                yb = yb.to(device)

                pred = self.model(xb)

                score = accuracy_metric(
                    pred,
                    yb
                )

                scores.extend(
                    score.cpu().numpy().tolist()
                )

                count += len(xb)

        if not scores:
            return 0.0

        return float(
            np.mean(scores)
        )

    def pretrain(self, train_dataset, test_dataset):

        print("\n🚀 Начинаю предобучение...")

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available()
        )

        best_score = -float("inf")

        best_state = None

        best_optimizer_state = None

        best_epoch = 0

        no_improve = 0

        for epoch in range(
            1,
            PRETRAIN_EPOCHS + 1
        ):

            if self.stop_event.is_set():

                print(
                    "⏹️ Предобучение остановлено."
                )

                break

            self.model.train()

            total_loss = 0.0

            batches = 0

            for x, y in train_loader:

                if self.stop_event.is_set():

                    break

                x = x.to(
                    device,
                    non_blocking=True
                )

                y = y.to(
                    device,
                    non_blocking=True
                )

                self.optimizer.zero_grad(
                    set_to_none=True
                )

                pred = self.model(x)

                loss = compute_loss(pred, y)

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    5.0
                )

                self.optimizer.step()

                total_loss += loss.item()

                batches += 1

            if self.stop_event.is_set():

                print(
                    "⏹️ Предобучение остановлено."
                )

                break

            avg_loss = (
                total_loss /
                max(1, batches)
            )

            # ==========================================
            # 📊 Оценка эпохи
            # ==========================================

            test_score = self.evaluate(
                test_dataset
            )

            print(
                f"📚 Epoch "
                f"{epoch}/{PRETRAIN_EPOCHS} | "
                f"loss={avg_loss:.6f} | "
                f"test={test_score:.2f}%"
            )

            # ==========================================
            # 💾 Сохраняем последнюю завершённую эпоху
            # ==========================================

            torch.save(
                {
                    "model":
                        self.model.state_dict(),

                    "optimizer":
                        self.optimizer.state_dict(),

                    "epoch":
                        epoch,

                    "test_score":
                        test_score,

                    "best_score":
                        best_score,

                    "threshold":
                        ACC_THRESHOLD,

                    "hidden":
                        HIDDEN,

                    "mid":
                        H_MID,

                    "params":
                        parameter_count,
                },
                "/content/forecaster_last.pth"
            )

            print(
                "💾 Последняя эпоха сохранена."
            )

            # ==========================================
            # 🏆 Новый лучший результат
            # ==========================================

            if test_score > best_score:

                best_score = test_score

                best_epoch = epoch

                no_improve = 0

                # Копия модели
                best_state = {
                    k: v.detach().clone()
                    for k, v
                    in self.model.state_dict().items()
                }

                # Копия оптимизатора
                best_optimizer_state = copy.deepcopy(
                    self.optimizer.state_dict()
                )

                # Сохраняем лучшую модель
                torch.save(
                    {
                        "model":
                            best_state,

                        "optimizer":
                            best_optimizer_state,

                        "epoch":
                            best_epoch,

                        "test_score":
                            best_score,

                        "threshold":
                            ACC_THRESHOLD,

                        "hidden":
                            HIDDEN,

                        "mid":
                            H_MID,

                        "params":
                            parameter_count,
                    },
                    MODEL_FILE
                )

                print(
                    f"🏆 НОВЫЙ ЛУЧШИЙ РЕЗУЛЬТАТ: "
                    f"{best_score:.2f}%"
                )

                print(
                    f"💾 Главная модель сохранена: "
                    f"{MODEL_FILE}"
                )

            else:

                no_improve += 1

                # ======================================
                # 🔄 Откат к лучшей модели
                # ======================================

                if best_state is not None:

                    self.model.load_state_dict(
                        best_state
                    )

                    self.optimizer.load_state_dict(
                        best_optimizer_state
                    )

                    print(
                        f"🔄 Результат хуже лучшего: "
                        f"{test_score:.2f}% < "
                        f"{best_score:.2f}%"
                    )

                    print(
                        f"↩️ Откат к эпохе "
                        f"{best_epoch}: "
                        f"{best_score:.2f}%"
                    )

            # ==========================================
            # ⏹️ Early stopping
            # ==========================================

            if no_improve >= 2:

                print(
                    "\n⏹️ Early stopping: "
                    "2 эпохи без улучшения."
                )

                break

        # ==============================================
        # 🏁 Возвращаем лучшую модель
        # ==============================================

        if best_state is not None:

            self.model.load_state_dict(
                best_state
            )

            self.optimizer.load_state_dict(
                best_optimizer_state
            )

        print(
            f"\n🏁 Предобучение завершено."
        )

        print(
            f"🏆 Лучший результат: "
            f"{best_score:.2f}%"
        )

        print(
            f"📚 Лучшая эпоха: "
            f"{best_epoch}"
        )

        print(
            f"💾 Главная модель: "
            f"{MODEL_FILE}"
        )

        print(
            "💾 Последняя эпоха: "
            "/content/forecaster_last.pth"
        )

    def fix_example(
        self,
        x,
        y
    ):

        if self.stop_event.is_set():
            return

        before = self.predict(x)

        before_score = accuracy_metric(
            torch.from_numpy(before).unsqueeze(0),
            torch.from_numpy(y).unsqueeze(0)
        )[0].item()

        xb = torch.from_numpy(
            x
        ).unsqueeze(0).float()

        yb = torch.from_numpy(
            y
        ).unsqueeze(0).float()

        for _ in range(FIX_STEPS):

            if self.stop_event.is_set():
                return

            self.train_batch(
                xb,
                yb,
                FIX_LR
            )

        after = self.predict(x)

        after_score = accuracy_metric(
            torch.from_numpy(after).unsqueeze(0),
            torch.from_numpy(y).unsqueeze(0)
        )[0].item()

        self.last_accuracy = after_score

        self.bad_buffer.append(
            (x.copy(), y.copy())
        )

        if len(self.bad_buffer) > BAD_BUF:

            self.bad_buffer.pop(0)

        self.fix_count += 1

        print(
            f"🔧 Fix #{self.fix_count}: "
            f"{before_score:.2f}% → "
            f"{after_score:.2f}%"
        )

        if (
            self.fix_count % MIX_EVERY == 0
            and len(self.bad_buffer) >= 2
        ):

            mix = random.sample(
                self.bad_buffer,
                min(
                    8,
                    len(self.bad_buffer)
                )
            )

            mx = torch.from_numpy(
                np.stack(
                    [a for a, b in mix]
                )
            ).float()

            my = torch.from_numpy(
                np.stack(
                    [b for a, b in mix]
                )
            ).float()

            for _ in range(MIX_STEPS):

                if self.stop_event.is_set():
                    return

                self.train_batch(
                    mx,
                    my,
                    MIX_LR
                )

    def save(self):

        torch.save(
            {
                "model": self.model.state_dict(),
                "optimizer": self.optimizer.state_dict(),
                "threshold": ACC_THRESHOLD,
                "hidden": HIDDEN,
                "mid": H_MID,
                "params": parameter_count,
            },
            MODEL_FILE
        )

        print(
            "💾 Модель сохранена:",
            MODEL_FILE
        )

    def load(self):

        if not os.path.exists(
            MODEL_FILE
        ):

            print(
                "Файл модели не найден."
            )

            return

        data = torch.load(
            MODEL_FILE,
            map_location=device
        )

        if isinstance(data, dict) and "model" in data:

            self.model.load_state_dict(
                data["model"]
            )

            if "optimizer" in data:

                try:
                    self.optimizer.load_state_dict(
                        data["optimizer"]
                    )
                except Exception:
                    pass

        else:

            self.model.load_state_dict(
                data
            )

        self.model.to(device)

        print(
            "📂 Модель загружена."
        )

def naive_baseline_score(dataset, max_samples=300):

    loader = DataLoader(dataset, batch_size=BATCH, shuffle=False)

    scores, count = [], 0

    with torch.no_grad():
        for xb, yb in loader:
            if count >= max_samples:
                break
            remaining = max_samples - count
            yb = yb[:remaining]

            pred = torch.zeros_like(yb)  # "ничего не изменилось"
            score = accuracy_metric(pred, yb)

            scores.extend(score.cpu().numpy().tolist())
            count += len(yb)

    return float(np.mean(scores)) if scores else 0.0

brain = Brain()

print("Brain готов.")

In [ ]:
# ============================================
# ЯЧЕЙКА 3
# Командный интерфейс
# ============================================

running_thread = None


def start_pretraining():

    global running_thread

    if running_thread is not None and running_thread.is_alive():

        print(
            "⚠️ Уже что-то выполняется."
        )

        return

    def worker():

        brain.pretrain(
            train_dataset,
            test_dataset
        )

    running_thread = threading.Thread(
        target=worker,
        daemon=True
    )

    running_thread.start()


def online_training():

    global running_thread

    if running_thread is not None and running_thread.is_alive():

        print(
            "⚠️ Уже что-то выполняется."
        )

        return

    def worker():

        brain.reset_stop()
        brain.phase = "ONLINE"

        print()
        print("=" * 60)
        print("🧠 ОНЛАЙН-РЕЖИМ")
        print("=" * 60)
        print(
            "В этом режиме используются "
            "последние данные CSV."
        )

        dataset = CandleDataset(
            G,
            o,
            h,
            l,
            c,
            0,
            len(c),
            STEP
        )

        if len(dataset) == 0:

            print(
                "Недостаточно данных."
            )

            brain.phase = "IDLE"
            return

        # Последние доступные примеры
        position = 0

        while not brain.stop_event.is_set():

            position = min(
                position,
                len(dataset) - 1
            )

            x, y = dataset[position]

            x_np = x.numpy()
            y_np = y.numpy()

            pred = brain.predict(
                x_np
            )

            score = accuracy_metric(
                torch.from_numpy(pred).unsqueeze(0),
                torch.from_numpy(y_np).unsqueeze(0)
            )[0].item()

            brain.last_accuracy = score

            print()
            print(
                f"📊 Online score: "
                f"{score:.2f}%"
            )

            if score < ACC_THRESHOLD:

                print(
                    f"❌ Ниже порога "
                    f"{ACC_THRESHOLD:.2f}%"
                )

                brain.fix_example(
                    x_np,
                    y_np
                )

            else:

                print(
                    f"✅ Выше порога "
                    f"{ACC_THRESHOLD:.2f}%"
                )

            position += 1

            if position >= len(dataset):

                position = 0

            # Небольшая пауза, чтобы не крутить
            # онлайн-обучение без остановки.
            for _ in range(10):

                if brain.stop_event.is_set():
                    break

                time.sleep(1)

        brain.phase = "IDLE"

        print(
            "⛔ Онлайн-режим остановлен."
        )

    running_thread = threading.Thread(
        target=worker,
        daemon=True
    )

    running_thread.start()


def print_status():

    alive = (
        running_thread is not None
        and running_thread.is_alive()
    )

    print()
    print("=" * 60)
    print("СТАТУС")
    print("=" * 60)
    print("Устройство:", device)
    print("Фаза:", brain.phase)
    print("Поток работает:", alive)
    print("Порог:", ACC_THRESHOLD)
    print("Fix count:", brain.fix_count)
    print("Последняя точность:", brain.last_accuracy)
    print("Параметров:", f"{parameter_count:,}")
    print("=" * 60)


print("""
============================================
КОМАНДЫ
============================================

P       — предобучение
T       — онлайн-обучение
S       — остановить обучение
V       — статус
SAVE    — сохранить модель
LOAD    — загрузить модель
THR 32  — установить порог 32%
Q       — выйти из консоли

============================================
""")


while True:

    try:

        command = input(">>> ").strip()

    except EOFError:

        print(
            "Ввод завершён."
        )

        break

    if not command:
        continue

    cmd = command.upper()

    if cmd == "P":

        start_pretraining()

    elif cmd == "T":

        online_training()

    elif cmd in (
        "S",
        "STOP"
    ):

        brain.stop()

    elif cmd in (
        "V",
        "STATUS"
    ):

        print_status()

    elif cmd == "SAVE":

        brain.save()

    elif cmd == "LOAD":

        brain.load()

    elif cmd.startswith("THR"):

        parts = command.split()

        if len(parts) != 2:

            print(
                "Пример: THR 32"
            )

            continue

        try:

            ACC_THRESHOLD = float(
                parts[1]
            )

            print(
                f"Порог установлен: "
                f"{ACC_THRESHOLD}%"
            )

        except ValueError:

            print(
                "Неверное значение."
            )

    elif cmd == "Q":

        brain.stop()

        print(
            "Выход из консоли."
        )

        break

    else:

        print(
            "Неизвестная команда."
        )